In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

N_CUSTOMERS = 10_000

customers = pd.DataFrame({
    "customer_id": range(100001, 100001 + N_CUSTOMERS),
    "age": np.random.randint(21, 65, N_CUSTOMERS),
    "annual_income": np.random.lognormal(
        mean=np.log(700000),
        sigma=0.55,
        size=N_CUSTOMERS
    ),
    "employment_years": np.random.randint(0, 30, N_CUSTOMERS),
    "dependents": np.random.randint(0, 5, N_CUSTOMERS),
    "savings": np.random.lognormal(
        mean=np.log(250000),
        sigma=0.8,
        size=N_CUSTOMERS
    ),
    "existing_debt": np.random.lognormal(
        mean=np.log(150000),
        sigma=0.8,
        size=N_CUSTOMERS
    )
})

customers.head()

,customer_id,age,annual_income,employment_years,dependents,savings,existing_debt
0,100001,59,6.542970e+05,0,3,281262.935425,185332.880592
1,100002,49,6.216075e+05,4,3,189110.449367,116700.181303
2,100003,35,6.371762e+05,10,2,396507.236148,67320.829790
3,100004,63,1.915457e+06,19,4,350710.898806,217865.937714
4,100005,28,3.833640e+05,10,4,324423.445016,446028.336260


In [3]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   age               10000 non-null  int32  
 2   annual_income     10000 non-null  float64
 3   employment_years  10000 non-null  int32  
 4   dependents        10000 non-null  int32  
 5   savings           10000 non-null  float64
 6   existing_debt     10000 non-null  float64
dtypes: float64(3), int32(3), int64(1)
memory usage: 429.8 KB


In [4]:
customers["credit_score"] = np.clip(
    np.random.normal(700, 70, N_CUSTOMERS),
    300,
    850
).round().astype(int)

In [5]:
employment_types = [
    "Salaried",
    "Self-Employed",
    "Business Owner",
    "Contract"
]

customers["employment_type"] = np.random.choice(
    employment_types,
    size=N_CUSTOMERS,
    p=[0.55, 0.20, 0.15, 0.10]
)

In [6]:
customers["debt_to_income"] = (
    customers["existing_debt"] /
    customers["annual_income"]
)

customers["savings_to_income"] = (
    customers["savings"] /
    customers["annual_income"]
)

In [7]:
risk_score = (
    0.35 * (1 - customers["credit_score"] / 850)
    + 0.30 * customers["debt_to_income"].clip(0, 1)
    + 0.20 * (1 - customers["savings_to_income"].clip(0, 1))
    + 0.15 * (1 - customers["employment_years"] / 30)
)

risk_score = np.clip(risk_score, 0, 1)

In [8]:
default_probability = 1 / (
    1 + np.exp(-8 * (risk_score - 0.45))
)

customers["default_probability"] = default_probability

In [9]:
customers["default"] = (
    np.random.random(N_CUSTOMERS)
    < customers["default_probability"]
).astype(int)

In [10]:
customers["default"].value_counts(normalize=True)

default
0    0.6781
1    0.3219
Name: proportion, dtype: float64

In [11]:
customers.to_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/database/customers.csv",
    index=False
)

In [13]:
pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/database/customers.csv").head()

,customer_id,age,annual_income,employment_years,dependents,savings,existing_debt,credit_score,employment_type,debt_to_income,savings_to_income,default_probability,default
0,100001,59,6.542970e+05,0,3,281262.935425,185332.880592,708,Contract,0.283255,0.429870,0.415757,1
1,100002,49,6.216075e+05,4,3,189110.449367,116700.181303,810,Salaried,0.187739,0.304228,0.296416,1
2,100003,35,6.371762e+05,10,2,396507.236148,67320.829790,595,Salaried,0.105655,0.622288,0.249349,1
3,100004,63,1.915457e+06,19,4,350710.898806,217865.937714,761,Salaried,0.113741,0.183095,0.216395,1
4,100005,28,3.833640e+05,10,4,324423.445016,446028.336260,689,Business Owner,1.163459,0.846254,0.592992,0
